In this notebook, we are going to load some of the data that we processed in the Condensed Notebook and perform across-trial analysis of the power spectra using FOOOF, an iterative peak-detection and PSD 1/f fitting algorithm. 

Also provide options for plotting the frequency-wide and session-wide PSD by region.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import mne
import mne_connectivity
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp
import pandas as pd
from mne.preprocessing.bads import _find_outliers
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 
from ast import literal_eval


# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest

from neurodsp.plts.spectral import plot_power_spectra


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')
sys.path.append('/hpc/users/qasims01/resources/abcTau')


In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


Before running the following, you need to: 

1. Load relevant behavioral data produced from your proprietary behavioral analyses.
1. Load neural epochs of behavioral interest produced from the Condensed Notebook.

## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

## Plot the power spec by region

Before using FOOOF to compute peak properties constrained to lower frequencies, let's first just plot the power spectra in a region-specific manner, normalizing for comparison across participants: 

In [ ]:
%%time
# select event of interest
ev_dict = {'feedback_start': [0, 1.5]}

# select ROI
rois = ['HPC', 'AMY', 'OFC', 'dmPFC', 'ACC'] 

progress_bar = tqdm(subj_ids, ascii=True, desc='Computing PSD for all trials and subsetting by conditions')

psd_group_data = {f'{a}': [] for a in rois}

for subj_id in progress_bar:
    # Load one person's data 
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            if not os.path.exists(f'{filepath}/plots'):
                os.makedirs(f'{filepath}/plots')
            if not os.path.exists(f'{filepath}/dfs'):
                os.makedirs(f'{filepath}/dfs')

            mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)
            mne_data_reref.metadata = learn_df[learn_df.participant==subj_id]
            
            mne_data_reref.crop(tmin=ev_dict[event][0], tmax=ev_dict[event][-1])
            
            # NAN out the bad data
            # THE following will now LOAD in dataframes that indicate IED and artifact time points in your data
            IED_df = pd.read_csv(f'{load_path}/{event}_IED_df.csv') 
            artifact_df = pd.read_csv(f'{load_path}/{event}_artifact_df.csv') 

            # Now, let's iterate through each channel, and each ied/artifact, and NaN 100 ms before and after these timepoints
            for ch_ix, ch_name in enumerate(mne_data_reref.ch_names): 
                ied_ev_list = IED_df[ch_name].dropna().index.tolist()
                artifact_ev_list = artifact_df[ch_name].dropna().index.tolist() 
                for ev_ in ied_ev_list: 
                    for ied_ in literal_eval(IED_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((ied_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((ied_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan
                for ev_ in artifact_ev_list: 
                    for artifact_ in literal_eval(artifact_df[ch_name].iloc[ev_]):
                        # remove 100 ms before 
                        ev_ix_start = np.max([0, np.floor((artifact_- 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        # remove 100 ms after
                        ev_ix_end = np.min([mne_data_reref._data.shape[-1], np.ceil((artifact_ + 0.1) * mne_data_reref.info['sfreq'])]).astype(int)
                        mne_data_reref._data[ev_, ch_ix, ev_ix_start:ev_ix_end] = np.nan


            save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            # Get electrode df 
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
                    
            progress_bar = tqdm(elec_df.label, ascii=True, desc='Computing PSDs')

            for elec_ix, chan in enumerate(progress_bar):
                region = elec_df.salman_region.iloc[elec_ix]
                if region in rois:
                    # compute PSD here 
                    epo_spectrum = mne_data_reref.copy().pick([chan]).compute_psd(method='multitaper',
                                                fmin=2, 
                                                fmax=200,
                                                tmin=ev_dict[event][0],
                                                tmax=ev_dict[event][-1],
                                                verbose=False)
                                                
#                     psds, freqs = epo_spectrum.get_data(return_freqs=True)
                    psds = epo_spectrum._data
                    freqs = epo_spectrum.freqs
                    # average across trials and squeeze out the empty axis 
                    psd_group_data[region].append(np.nanmean(psds, axis=0)[0,:])
                    
                    
                    
                    

the following function is just useful for normalizing the PSDs to the range of 0-1

In [ ]:
def normalize_list(input_list):
    min_val = np.nanmin(input_list)
    max_val = np.nanmax(input_list)
    den = (max_val - min_val)
    if den == 0: 
        den = 0.001
    
    normalized_list = [(x - min_val) / den  for x in input_list]
    
    return normalized_list

In [ ]:
# freqs = np.logspace(*np.log10([2, 200]), num=30)
save_dir_plots = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots'

for region in rois: 

    fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
    # Create a Pandas DataFrame
    plot_data = np.vstack(psd_group_data[region])
#     normalized_array = np.array([normalize_list(lst) for lst in plot_data])
    for ch in plot_data:
        # plt.plot(freqs, np.log10(ch), alpha=0.1, color='k')
        plot_power_spectra(freqs, ch, alpha =0.1,  
                       ax=ax, colors='black')   
    plot_power_spectra(freqs, np.nanmean(plot_data, axis=0), 
                       ax=ax, colors='black')   
    # plt.plot(freqs, np.log10(np.nanmean(normalized_array, axis=0)), alpha=1, color='k')

    # ax.set(title=f'{region}_nelec={n_elecs}')
    sns.despine()
    
    plt.savefig(f'{save_dir_plots}/feedback_PSD_{region}.pdf', dpi=300)

        
        

## Power spectrum (FOOOF)

Here, we are going to plot the group-level FOOOF analysis for two conditions in the task across all channels, and then also plot the individual power spectrums for each channel with model fit for each of the two conditions. Note the time specification to make sure I leave out the baseline period... 

FOOOF's primary utility, imo, is not in peak parameterization but 1/f subtraction. Peak parameters are subject to over- and under-fitting, and fit quality has too strong of an effect. Simply contrasting *n* PSDs (for *n* conditions) after removing the 1/f is a more healthy approach imo.

For guidance on picking the parameters used to fit: https://fooof-tools.github.io/fooof/auto_tutorials/plot_07-TroubleShooting.html#sphx-glr-auto-tutorials-plot-07-troubleshooting-py

## 1. FOOOF (produce dataframe and condition-specific plots)

In [ ]:
%%time
ev_dict = {'feedback_start': [0, 0.5]}

# Define peak_width, peak_height, peak_threshold, max_n_peaks
FOOOF_kwargs = {'peak_width_limits': [1, 8], 
               'min_peak_height': 0.0,
                'peak_threshold': 2.,
                'max_n_peaks': 3,
               'freq_range': [4, 30]}
 
# Define our frequency bands of interest
band_dict = {'theta' : [4, 8],
               'alpha' : [8, 13],
               'beta' : [13, 30]}

conditions = ['rpe>0',
             'rpe<1']

progress_bar = tqdm(subj_ids, ascii=True, desc='Computing FOOOF for all trials and subsetting by conditions')

for subj_id in progress_bar:
    # Load one person's data 
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            if not os.path.exists(f'{filepath}/plots'):
                os.makedirs(f'{filepath}/plots')
            if not os.path.exists(f'{filepath}/dfs'):
                os.makedirs(f'{filepath}/dfs')

            mne_data_reref = mne.read_epochs(f'{load_path}/{event}-epo.fif', preload=True)
            
            # mne_data_reref.metadata = learn_df[learn_df.participant==subj_id]
            # mne_data_reref.metadata = s

            save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            # Get electrode df 
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
            
            # # Run the FOOOF code in parallel
            Parallel(n_jobs=-1, verbose=1)(delayed(analysis_utils.compute_FOOOF_parallel)(chan, mne_data_reref, 
                                                                                         subj_id, 
                                                                                         elec_df, 
                                                                                         event, 
                                                                                         ev_dict, 
                                                                                         band_dict, 
                                                                                         conditions, 
                                                                                         do_plot=True, 
                                                                                         save_path='/sc/arion/projects/guLab/Salman/EphysAnalyses',
                                                                                         do_save=True, 
                                                                                         **FOOOF_kwargs) for chan in mne_data_reref.ch_names)


In [ ]:
ev_dict = {'feedback_start': [0, 1.5]}


With 25 cores, this took ~0.5 second / electrode

In [ ]:
# Load the dataframes: 
chan_dfs = [] 
for subj_id in subj_ids[0:1]:
    for day in ['Day1']:
        load_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        for event in ev_dict.keys():   
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/FOOOF/{event}'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
            for chan in tqdm(elec_df.label.unique(), ascii=True, desc='Loading FOOOF dataframes'): 
                chan_df = pd.read_csv(f'{filepath}/dfs/{chan}_df.csv')
                chan_dfs.append(chan_df)
chan_dfs = pd.concat(chan_dfs)



In [ ]:
oneTauOU_oneF(theta, deltaT, binSize, T, numTrials, data_mean, data_var)



In [ ]:
# Label the frequency bands for easy indexing: 

chan_dfs['fband'] = chan_dfs.apply(lambda x: 'theta' if ((x.frequency>=2) & (x.frequency<8)) else 'alpha' if ((x.frequency>=8) & (x.frequency<13)) else 'beta' if ((x.frequency>=13) & (x.frequency<=30)) else np.nan, axis=1)



Here I want to identify all the "theta electrodes" - electrodes with significant peaks in the theta range

In [ ]:
roi_df = chan_dfs[chan_dfs['region'].isin(['HPC', 'AMY', 'OFC'])]
roi_df = roi_df[(roi_df.cond=='all')]
roi_df['theta_peak'] = 0

In [ ]:
# these are the electrodes with >= 1 peak in theta frequencies
theta_channel_ids = roi_df[((roi_df['peak_freq'] > 2 ) & (roi_df['peak_freq'] < 8))][['participant', 'channel']].drop_duplicates()

for row in theta_channel_ids.itertuples():
    roi_df.loc[((roi_df.participant==row.participant) & (roi_df.channel==row.channel)), 'theta_peak'] = 1

In [ ]:
# roi_df['log_psd'] = np.log10(roi_df['PSD_raw'])
# roi_df['log_freq'] = np.log10(roi_df['frequency'])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4), dpi=300, sharey=True)

sns.lineplot(data=roi_df[roi_df.region=='AMY'], 
             x='frequency', 
             y='PSD_raw', 
             errorbar='se', 
             hue='theta_peak', 
             ax=ax[0])
sns.lineplot(data=roi_df[roi_df.region=='HPC'], 
             x='frequency', 
             y='PSD_raw', 
             errorbar='se', 
             hue='theta_peak', 
             ax=ax[1])
sns.lineplot(data=roi_df[roi_df.region=='OFC'], 
             x='frequency', 
             y='PSD_raw', 
             errorbar='se', 
             hue='theta_peak', 
             ax=ax[2])

fig.tight_layout()
sns.despine()


In [ ]:
roi_df[['participant', 'region']].drop_duplicates().shape


In [ ]:
roi_df[((roi_df['peak_freq'] > 2 ) & (roi_df['peak_freq'] < 8))][['participant', 'region']].drop_duplicates().shape



In [ ]:
theta_channel_ids

In [ ]:
theta_channel_ids.to_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/theta_channel_df', index=None)


In [ ]:
for col in ['in_FOOOF_peak', 'peak_freq', 'peak_height']:
    roi_df[col] = roi_df[col].apply(lambda x: np.nan if x==0 else x) 


In [ ]:
plot_df = roi_df[(roi_df.fband=='theta') & (roi_df.theta_peak==1)].groupby(['participant', 'channel', 'region']).mean().reset_index()

plot_df


In [ ]:
ax = sns.jointplot(data = plot_df, 
                   x='PSD_exp', y='peak_freq', height=8, hue='region', ratio=2)
 
# plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# # get the current positions of the joint ax and the ax for the marginal x
# pos_joint_ax = ax.ax_joint.get_position()
# pos_marg_x_ax = ax.ax_marg_x.get_position()
# # reposition the joint ax so it has the same width as the marginal x ax
# ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# # reposition the colorbar using new x positions and y positions of the joint ax
# ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])

In [ ]:
grouped_df = chan_dfs.groupby(['region', 'channel', 'cond', 'in_FOOOF_peak']).mean().reset_index().drop(columns=['in_FOOOF_peak', 'frequency', 'PSD_raw'])


                                                                                 
                                                                                 

In [ ]:
grouped_df[grouped_df.peak_freq!=0]

In [ ]:
# # Fit an individual channel (for plotting)
# ind_fits = FOOOFGroup_res.get_fooof(ind=0, regenerate=True)
# ind_fits.fit()

In [ ]:
# ind_fits.plot()